In [20]:
import pandas as pd
import numpy as np

df = pd.read_csv("play_tennis_dataset.csv")
df["PlayTennis"] = df["PlayTennis"].str.strip()
df["Outlook"] = df["Outlook"].str.strip()
df["Temperature"] = df["Temperature"].str.strip()
df["Humidity"] = df["Humidity"].str.strip()
df["Wind"] = df["Wind"].str.strip()
df = df.drop(columns=["Day"])


print(df)

     Outlook Temperature Humidity    Wind PlayTennis
0      Sunny         Hot     High    Weak         No
1      Sunny         Hot     High  Strong         No
2   Overcast         Hot     High    Weak        Yes
3       Rain        Mild     High    Weak        Yes
4       Rain        Cool   Normal    Weak        Yes
5       Rain        Cool   Normal  Strong         No
6   Overcast        Cool   Normal  Strong        Yes
7      Sunny        Mild     High    Weak         No
8      Sunny        Cool   Normal    Weak        Yes
9       Rain        Mild   Normal    Weak        Yes
10     Sunny        Mild   Normal  Strong        Yes
11  Overcast        Mild     High  Strong        Yes
12  Overcast         Hot   Normal    Weak        Yes
13      Rain        Mild     High  Strong         No


In [21]:
def entropy(target_col):
    elements, counts = np.unique(target_col, return_counts=True)
    entropy_val = 0
    for i in range(len(elements)):
        prob = counts[i] / np.sum(counts)
        entropy_val += -prob * np.log2(prob)
    return entropy_val


print("Entropy:", entropy(df["PlayTennis"]))

Entropy: 0.9402859586706311


In [22]:
def info_gain(data, split_attribute, target_name="PlayTennis"):
    total_entropy = entropy(data[target_name])
    vals, counts = np.unique(data[split_attribute], return_counts=True)
    weighted_entropy = 0
    for i in range(len(vals)):
        subset = data[data[split_attribute] == vals[i]]
        weighted_entropy += (counts[i] / np.sum(counts)) * entropy(subset[target_name])
    return total_entropy - weighted_entropy


features = ["Outlook", "Temperature", "Humidity", "Wind"]
for f in features:
    gain = info_gain(df, f)
    print(f"{f}: {gain:.3f}")

Outlook: 0.247
Temperature: 0.029
Humidity: 0.152
Wind: 0.048


In [23]:
def id3(data, features, target_name="PlayTennis"):

    if len(np.unique(data[target_name])) == 1:
        return np.unique(data[target_name])[0]


    if len(features) == 0:
        return data[target_name].mode()[0]


    gains = [info_gain(data, f, target_name) for f in features]
    best_feature = features[np.argmax(gains)]


    tree = {best_feature: {}}

    for val in np.unique(data[best_feature]):
        sub_data = data[data[best_feature] == val].drop(columns=[best_feature])
        new_features = [f for f in features if f != best_feature]
        subtree = id3(sub_data, new_features, target_name)
        tree[best_feature][val] = subtree

    return tree

In [19]:
features = [col for col in df.columns if col != "PlayTennis"]
decision_tree = id3(df, features)
print("\nDecision Tree:\n", decision_tree)


Decision Tree:
 {'Outlook': {'Overcast': 'Yes', 'Rain': {'Wind': {'Strong': 'No', 'Weak': 'Yes'}}, 'Sunny': {'Humidity': {'High': 'No', 'Normal': 'Yes'}}}}
